In [ ]:
# 1. Initialization, GCP AUTH, & Dataset Creation
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import pytz
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')

# Memastikan akun tersambung
auth.authenticate_user()

project_id = 'fsb-12345'
dataset_id_slv = 'slv_bike_store'
dataset_id_gld = 'gld_bike_store'

client = bigquery.Client(project=project_id)

# Membuat dataset Gold
dataset_ref = bigquery.Dataset(f'{project_id}.{dataset_id_gld}')
dataset_ref.location = "US"
dataset_gld = client.create_dataset(dataset_ref, exists_ok=True)
print(f"✅ Dataset '{dataset_id_gld}' siap digunakan untuk Reporting (Gold Layer).")

# Helper functions (Re-use & Sinkronisasi metadata)
def extract_data_from_gbq(query):
    return client.query(query).to_dataframe()

from pandas_gbq import to_gbq
def load_data_to_gbq(data, table_name, id_dataset):
    destination = f'{id_dataset}.{table_name}'
    for col in data.columns:
        if data[col].dtype == 'object':
            data[col] = data[col].astype(str)
    to_gbq(data, destination_table=destination, project_id=project_id, if_exists='replace', progress_bar=False)
    print(f"   [SUCCESS] Tabel {destination} berhasil dimuat.")

def add_metadata_gold(data):
    now_utc = datetime.now(timezone.utc)
    date_now = now_utc.astimezone(pytz.timezone('Asia/Jakarta'))
    data_add_md = data.copy()
    data_add_md['job_gold_date'] = pd.to_datetime(date_now)
    data_add_md['job_gold_by'] = 'SYSTEM_GOLD'
    return data_add_md

✅ Dataset 'gld_bike_store' siap digunakan untuk Reporting (Gold Layer).


In [ ]:
# 2. Proses Fact Table
print("\n=== 1. Membentuk Fact Table (Detail Transaksi) ===")

# Menyusun detail transaksi dengan metrik terukur (quantity, price, discount, total_price)
query_fact_detail = f"""
  SELECT
    o.order_id,
    oi.order_item_id,
    o.customer_id,
    CAST(o.order_date AS DATE) AS order_date,
    CAST(o.required_date AS DATE) AS required_date,
    CAST(o.shipped_date AS DATE) AS shipped_date,
    o.store_id,
    oi.product_id,
    st.staff_id,
    oi.quantity,
    oi.list_price AS product_price,
    oi.discount,
    ROUND(oi.quantity * oi.list_price * (1 - oi.discount), 2) AS total_price
  FROM {project_id}.{dataset_id_slv}.orders o
  LEFT JOIN {project_id}.{dataset_id_slv}.order_items oi ON o.order_id = oi.order_id
  LEFT JOIN {project_id}.{dataset_id_slv}.staffs st ON st.staff_id = o.staff_id
"""

detail_transaction = extract_data_from_gbq(query_fact_detail)
detail_transaction_md = add_metadata_gold(detail_transaction)

# Load Fact Detail ke Gold Dataset
load_data_to_gbq(
    data=detail_transaction_md,
    table_name='fact_detail_transaction',
    id_dataset=dataset_id_gld
)


=== 1. Membentuk Fact Table (Detail Transaksi) ===
   [SUCCESS] Tabel gld_bike_store.fact_detail_transaction berhasil dimuat.


In [ ]:
pip install ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.5 MB/s eta 0:00:00


In [ ]:
# 3. Data Profiling
print("\n=== 2. Menjalankan Data Profiling ===")
try:
    from ydata_profiling import ProfileReport
    profile_detail_transaction = ProfileReport(detail_transaction, title='Profiling Report Data Detail Transaction', minimal=True)
    profile_detail_transaction.to_file('profile_detail_transaction.html')
    print("   [SUCCESS] File 'profile_detail_transaction.html' berhasil disimpan.")
except Exception as e:
    print(f"   [Warning] Gagal melakukan profiling: {e}. Pastikan ydata-profiling sudah terinstall.")


=== 2. Menjalankan Data Profiling ===


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 13/13 [00:00<00:00, 42.94it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

   [SUCCESS] File 'profile_detail_transaction.html' berhasil disimpan.


In [ ]:
# 4. Proses Fact Table
print("\n=== 3. Membentuk Fact Table (Summary Transaksi) ===")

summary_transaction = detail_transaction.groupby(
    by=['order_date', 'store_id', 'product_id', 'customer_id', 'staff_id'],
    as_index=False
).agg(
    total_order_items=('order_item_id', 'count'),
    total_revenue=('total_price', 'sum')
)

summary_transaction_md = add_metadata_gold(summary_transaction)

load_data_to_gbq(
    data=summary_transaction_md,
    table_name='fact_summary_transaction',
    id_dataset=dataset_id_gld
)


=== 3. Membentuk Fact Table (Summary Transaksi) ===
   [SUCCESS] Tabel gld_bike_store.fact_summary_transaction berhasil dimuat.


In [ ]:
# 5. Proses Dimension Tables
print("\n=== 4. Memproses Dimension Tables (Star Schema) ===")

# Buat query dimensi yang kaya informasi
dim_queries = {
    'dim_customers': f"SELECT customer_id, CONCAT(first_name, ' ', last_name) AS customer_name, phone, email, street, city, state, zip_code FROM {project_id}.{dataset_id_slv}.customers",
    'dim_stores': f"SELECT store_id, store_name, phone, email, street, city, state, zip_code FROM {project_id}.{dataset_id_slv}.stores",
    'dim_staffs': f"SELECT staff_id, CONCAT(first_name, ' ', last_name) AS staff_name, email, phone, active, store_id, manager_id FROM {project_id}.{dataset_id_slv}.staffs",
    'dim_products': f"""
        SELECT
            p.product_id, p.product_name, p.model_year, p.list_price,
            b.brand_name, c.category_name
        FROM {project_id}.{dataset_id_slv}.products p
        LEFT JOIN {project_id}.{dataset_id_slv}.brands b ON p.brand_id = b.brand_id
        LEFT JOIN {project_id}.{dataset_id_slv}.categories c ON p.category_id = c.category_id
    """
}

for table_name, query_str in dim_queries.items():
    print(f"\n⚙️ Membuat tabel dimensi: {table_name}")
    try:
        # Ekstrak data dimensi
        dim_data = extract_data_from_gbq(query_str)

        # Tambahkan metadata gold
        dim_data_md = add_metadata_gold(dim_data)

        # Memasukkan 'dim_data_md'
        load_data_to_gbq(
            data=dim_data_md,
            table_name=table_name,
            id_dataset=dataset_id_gld
        )
    except Exception as e:
        print(f"❌ Gagal memproses dimensi {table_name}. Error: {e}")

print("\n=== 🥇 Pipeline Gold Layer Selesai Seluruhnya! Data Siap Dihubungkan ke Looker Studio. ===")


=== 4. Memproses Dimension Tables (Star Schema) ===

⚙️ Membuat tabel dimensi: dim_customers
   [SUCCESS] Tabel gld_bike_store.dim_customers berhasil dimuat.

⚙️ Membuat tabel dimensi: dim_stores
   [SUCCESS] Tabel gld_bike_store.dim_stores berhasil dimuat.

⚙️ Membuat tabel dimensi: dim_staffs
   [SUCCESS] Tabel gld_bike_store.dim_staffs berhasil dimuat.

⚙️ Membuat tabel dimensi: dim_products
   [SUCCESS] Tabel gld_bike_store.dim_products berhasil dimuat.

=== 🥇 Pipeline Gold Layer Selesai Seluruhnya! Data Siap Dihubungkan ke Looker Studio. ===
